# Установка библиотек

In [1]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import base64

/Users/anlimka/Desktop/МЛ/Labolatornye_ML/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Загрузка и обработка данных

In [2]:
def load_and_preprocess_data(file_path='train.csv', nrows=None):
    data = pd.read_csv(file_path, sep=",")
    
    cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    existing_cols = [col for col in cols if col in data.columns]
    data[existing_cols] = data[existing_cols].fillna(data[existing_cols].mean())
   
    categorical_cols = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']
    for col in categorical_cols:
        if col in data.columns:
            mode_val = data[col].mode()[0]
            data[col] = data[col].fillna(mode_val)
    
    if 'Cabin' in data.columns:
        data[['Deck', 'CabinNumber', 'Side']] = data['Cabin'].str.split('/', expand=True)
        data['CabinNumber'] = pd.to_numeric(data['CabinNumber'])
    
    data_encoded = pd.get_dummies(
        data,
        columns=['HomePlanet', 'Destination'],
        prefix=['HomePlanet', 'Destination']
    )
    
    cols_to_drop = ['Cabin', 'PassengerId', 'Name']
    existing_drop = [col for col in cols_to_drop if col in data_encoded.columns]
    data_encoded = data_encoded.drop(columns=existing_drop)
    
    if 'Deck' in data_encoded.columns and 'Side' in data_encoded.columns:
        oe = OrdinalEncoder()
        data_encoded[['Deck', 'Side']] = oe.fit_transform(data_encoded[['Deck', 'Side']])
    
    if 'Transported' in data_encoded.columns:
        col = data_encoded.pop('Transported')
        data_encoded['Transported'] = col
    
    X = data_encoded.drop(columns=['Transported'])
    y = data_encoded['Transported'].astype(int)
    
    return X, y, data_encoded


X, y, data_processed = load_and_preprocess_data('train.csv')
print(f"Данные загружены: X.shape = {X.shape}, y.shape = {y.shape}")

Данные загружены: X.shape = (8693, 17), y.shape = (8693,)


# Масштабирование

In [3]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Кросс валидация и обучение

In [4]:
def train_decision_tree(max_depth, min_samples_split, min_samples_leaf, criterion, cv_folds):
    
    model = DecisionTreeClassifier(
        max_depth=max_depth if max_depth > 0 else None,
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        criterion=criterion,
        random_state=42
    )
    
    cv_scores = cross_val_score(
        model, X_scaled, y, 
        scoring='accuracy', 
        cv=int(cv_folds)
    )
    
    model.fit(X_scaled, y)
    
    # Важность признаков
    feature_importance = dict(zip(X.columns, model.feature_importances_))
    sorted_features = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))
    
    # Результаты кросс-валидации
    cv_scores_dict = {f'Fold_{i+1}': float(cv_scores[i]) for i in range(len(cv_scores))}
    
    return (
        cv_scores_dict,                      
        cv_scores_dict,                      
        float(np.mean(cv_scores)),           
        float(np.std(cv_scores)),            
        sorted_features,                     
        model                                
    )

# График важности признаков

In [5]:
def plot_feature_importance(feature_importance_dict):
    fig, ax = plt.subplots(figsize=(10, 6))
    features = list(feature_importance_dict.keys())[:10]  # Топ-10
    importance = list(feature_importance_dict.values())[:10]
    
    bars = ax.barh(range(len(features)), importance, color='pink')
    ax.set_yticks(range(len(features)))
    ax.set_yticklabels(features)
    ax.set_xlabel('Важность')
    ax.set_title('Топ-10 важности признаков (Feature Importance)')
    ax.invert_yaxis()
    
    for i, (idx, val) in enumerate(zip(range(len(features)), importance)):
        ax.text(val + 0.01, i, f'{val:.3f}', va='center')
    
    plt.tight_layout()
    return fig


# Делеyие на выборки и метрики

In [6]:
def test_on_split(model, test_size):
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    model_copy = DecisionTreeClassifier(**model.get_params())
    model_copy.fit(X_train, y_train)
    y_pred = model_copy.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm)
    ax_cm.set_xlabel('Предсказанные')
    ax_cm.set_ylabel('Истинные')
    ax_cm.set_title('Матрица ошибок')

    report = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    
    return accuracy, fig_cm, report_df

# Результаты

In [7]:
def analyze_data(max_depth, min_samples_split, min_samples_leaf, criterion, cv_folds, test_size):
  
    cv_scores, cv_scores_kv, mean_acc, std_acc, feature_importance, model = train_decision_tree(
        max_depth, min_samples_split, min_samples_leaf, criterion, cv_folds
    )

    importance_fig = plot_feature_importance(feature_importance)

    test_acc, cm_fig, report_df = test_on_split(model, test_size)
    
    result_text = f"""
    ### РЕЗУЛЬТАТЫ КРОСС-ВАЛИДАЦИИ:
    - Средняя точность : **{mean_acc:.4f}**
    - Стандартное отклонение: **{std_acc:.4f}**
    - Минимальная точность по фолдам: **{min(cv_scores.values()):.4f}**
    - Максимальная точность по фолдам: **{max(cv_scores.values()):.4f}**
    
    ### РЕЗУЛЬТАТЫ НА ОТДЕЛЬНОЙ ТЕСТОВОЙ ВЫБОРКЕ ({test_size*100:.0f}% данных):
    - Точность : **{test_acc:.4f}**
    
    ### ПАРАМЕТРЫ МОДЕЛИ:
    - max_depth: {max_depth if max_depth > 0 else 'None (без ограничений)'}
    - min_samples_split: {int(min_samples_split)}
    - min_samples_leaf: {int(min_samples_leaf)}
    - criterion: {criterion}
    - Количество фолдов кросс-валидации: {int(cv_folds)}
    """
    if max_depth <= 6 or (max_depth == 0 and model.tree_.max_depth <= 5):
        from sklearn.tree import plot_tree
        fig_tree, ax_tree = plt.subplots(figsize=(20, 12))
        plot_tree(
            model, 
            feature_names=X.columns.tolist(),
            class_names=['Not Transported', 'Transported'] if hasattr(model, 'classes_') else None,
            filled=True, 
            rounded=True, 
            ax=ax_tree, 
            fontsize=8,
            proportion=False
        )
        plt.title(f"Дерево решений (глубина = {model.tree_.max_depth})")
        plt.tight_layout()
        
        tree_fig = fig_tree
    else:
        print(f'Дерево слишком глубокое для визуализаци')
    
    return result_text, cv_scores, mean_acc, importance_fig, cm_fig, report_df,tree_fig

In [11]:
inputs = [
    gr.Slider(minimum=0, maximum=20, step=1, value=5, label="max_depth (глубина дерева, 0 = без ограничений)"),
    gr.Slider(minimum=2, maximum=20, step=1, value=2, label="min_samples_split (мин. образцов для разбиения)"),
    gr.Slider(minimum=1, maximum=20, step=1, value=1, label="min_samples_leaf (мин. образцов в листе)"),
    gr.Dropdown(choices=["gini", "entropy"], value="gini", label="criterion (критерий качества)"),
    gr.Slider(minimum=3, maximum=10, step=1, value=5, label="Количество фолдов для кросс-валидации"),
    gr.Slider(minimum=0.1, maximum=0.4, step=0.05, value=0.2, label="Размер тестовой выборки (от общей)")
]

outputs = [
    gr.Markdown(label="📈 РЕЗУЛЬТАТЫ АНАЛИЗА"),
    gr.Label(label="Оценки по фолдам"),
    gr.Number(label="Средняя точность"),
    gr.Plot(label="Важность признаков"),
    gr.Plot(label="Матрица ошибок"),
    gr.Dataframe(label="Отчет классификации"),
    gr.Plot(label="Визуализация дерева решений")
]

In [12]:
iface = gr.Interface(
    fn=analyze_data,
    inputs=inputs,
    outputs=outputs,
    title="Дерево решений",
    description="""
    """,
    theme="soft"
)

/Users/anlimka/Desktop/МЛ/Labolatornye_ML/.venv/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [13]:
if __name__ == "__main__":
    iface.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
